<a href="https://colab.research.google.com/github/Sanjeev2004/LandscapePainting-in-Van-Gogh-style-using-GAN/blob/main/music_genre_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📦 Cell 1: Install Dependencies

In [ ]:
!pip install kaggle librosa xgboost scikit-learn tensorflow matplotlib seaborn -q

## 🔑 Cell 2: Kaggle API Setup & Dataset Download

In [ ]:
import os
from google.colab import files

# Step 1: kaggle.json upload karo (Kaggle → Account → API → Create New Token)
print('📂 kaggle.json upload karo (Kaggle > Account > Create New API Token):')
uploaded = files.upload()

# Step 2: kaggle.json sahi jagah rakhov
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle API configured!')

In [ ]:
# Step 3: GTZAN Dataset download karo
!kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification
!unzip -q gtzan-dataset-music-genre-classification.zip -d gtzan
print('✅ GTZAN dataset downloaded & extracted!')
!ls gtzan/Data/genres_original/

 Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, BatchNormalization,
                                      Flatten, Dense, Dropout, GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# ── CONFIG ──────────────────────────────────────────────
DATA_PATH   = 'gtzan/Data/genres_original'
GENRES      = sorted(os.listdir(DATA_PATH))
SR          = 22050      # Sample rate
DURATION    = 30         # seconds
N_MELS      = 128        # Mel-spectrogram height
N_MFCC      = 40         # MFCC features per frame
HOP_LENGTH  = 512
N_FFT       = 2048
IMG_H, IMG_W = 128, 128  # CNN input size
BATCH_SIZE  = 32
EPOCHS      = 50
# ────────────────────────────────────────────────────────

print(f'Genres found: {GENRES}')
print(f'Total genres: {len(GENRES)}')

## 🔬 Cell 4: Feature Extraction
### Ek audio file ka visualization

In [ ]:
# Sample audio visualize karo
sample_file = os.path.join(DATA_PATH, 'jazz', 'jazz.00000.wav')
y, sr = librosa.load(sample_file, sr=SR, duration=DURATION)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Waveform
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title('Waveform')

# Mel-Spectrogram
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP_LENGTH)
mel_db = librosa.power_to_db(mel, ref=np.max)
img_mel = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel', ax=axes[1], cmap='magma')
axes[1].set_title('Mel-Spectrogram')
plt.colorbar(img_mel, ax=axes[1])

# MFCC
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
img_mfcc = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[2], cmap='coolwarm')
axes[2].set_title('MFCC')
plt.colorbar(img_mfcc, ax=axes[2])

plt.suptitle('Jazz - Feature Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from tqdm import tqdm
from skimage.transform import resize
import os # Added import os

# ── CONFIG ──────────────────────────────────────────────
DATA_PATH   = 'gtzan/Data/genres_original'
GENRES      = sorted(os.listdir(DATA_PATH))
SR          = 22050      # Sample rate
DURATION    = 30         # seconds
N_MELS      = 128        # Mel-spectrogram height
N_MFCC      = 40         # MFCC features per frame
HOP_LENGTH  = 512
N_FFT       = 2048
IMG_H, IMG_W = 128, 128  # CNN input size
BATCH_SIZE  = 32
EPOCHS      = 50
# ────────────────────────────────────────────────────────

SEGMENT_DURATION = 10  # 30s audio → 3 segments of 10s each

def extract_features_segmented(file_path, n_segments=3):
    """Ek file se n_segments alag features nikalta hai"""
    y_full, sr = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)
    seg_len = len(y_full) // n_segments

    results = []
    for i in range(n_segments):
        y = y_full[i*seg_len : (i+1)*seg_len]

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                              n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db   = librosa.power_to_db(mel, ref=np.max)
        mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        mel_img  = resize(mel_norm, (IMG_H, IMG_W), anti_aliasing=True)[..., np.newaxis]

        mfcc    = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)
        chroma  = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP_LENGTH)
        contrast= librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP_LENGTH)
        zcr     = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
        rms     = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)

        xgb_vec = np.concatenate([
            np.mean(mfcc,axis=1), np.std(mfcc,axis=1), np.max(mfcc,axis=1),
            np.mean(chroma,axis=1), np.std(chroma,axis=1),
            np.mean(contrast,axis=1),
            [np.mean(zcr), np.std(zcr)],
            [np.mean(rms), np.std(rms)],
            [np.mean(rolloff), np.std(rolloff)]
        ])
        results.append((mel_img, xgb_vec))
    return results


# ── Loop replace karo ──────────────────────────────────
mel_images, xgb_features, labels = [], [], []

for genre in tqdm(GENRES, desc='Genres'):
    genre_path = os.path.join(DATA_PATH, genre)
    for fname in os.listdir(genre_path):
        if fname.endswith('.wav'):
            fpath = os.path.join(genre_path, fname)
            try:
                segments = extract_features_segmented(fpath, n_segments=3)
                for mel_img, xgb_feat in segments:
                    mel_images.append(mel_img)
                    xgb_features.append(xgb_feat)
                    labels.append(genre)
            except Exception as e:
                print(f'⚠️  Skipped {fname}: {e}')

## 🏗️ Cell 5: Full Feature Extraction (All Files)

In [ ]:
from tqdm import tqdm
from skimage.transform import resize

def extract_features(file_path):
    """
    Returns:
      mel_img  : (IMG_H, IMG_W, 1) — CNN ke liye
      mfcc_vec : (N_MFCC*3,)       — XGB ke liye (mean, std, max of each MFCC)
    """
    y, sr = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)

    # ── Mel-Spectrogram for CNN ──────────────────────────
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Normalize to [0,1]
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    # Resize to fixed (IMG_H x IMG_W)
    mel_img = resize(mel_norm, (IMG_H, IMG_W), anti_aliasing=True)
    mel_img = mel_img[..., np.newaxis]   # Add channel dim → (128,128,1)

    # ── MFCC features for XGBoost ───────────────────────
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)
    # Statistical aggregation: mean, std, max per MFCC coefficient
    mfcc_vec = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.max(mfcc, axis=1)
    ])  # → shape (120,)

    # ── Extra features for XGBoost ──────────────────────
    chroma   = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP_LENGTH)
    zcr      = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
    rms      = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    rolloff  = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)

    extra_vec = np.concatenate([
        np.mean(chroma, axis=1), np.std(chroma, axis=1),
        np.mean(contrast, axis=1),
        [np.mean(zcr), np.std(zcr)],
        [np.mean(rms), np.std(rms)],
        [np.mean(rolloff), np.std(rolloff)]
    ])

    xgb_features = np.concatenate([mfcc_vec, extra_vec])

    return mel_img, xgb_features


# ── Extract from all files ───────────────────────────────
mel_images, xgb_features, labels = [], [], []

for genre in tqdm(GENRES, desc='Genres'):
    genre_path = os.path.join(DATA_PATH, genre)
    for fname in os.listdir(genre_path):
        if fname.endswith('.wav'):
            fpath = os.path.join(genre_path, fname)
            try:
                mel_img, xgb_feat = extract_features(fpath)
                mel_images.append(mel_img)
                xgb_features.append(xgb_feat)
                labels.append(genre)
            except Exception as e:
                print(f'⚠️  Skipped {fname}: {e}')

mel_images   = np.array(mel_images,   dtype=np.float32)  # (N, 128, 128, 1)
xgb_features = np.array(xgb_features, dtype=np.float32)  # (N, features)
labels       = np.array(labels)

print(f'\nExtraction complete!')
print(f'   Mel-Spectrogram shape : {mel_images.shape}')
print(f'   XGB feature shape     : {xgb_features.shape}')
print(f'   Labels shape          : {labels.shape}')

## ✂️ Cell 6: Encode Labels & Train/Test Split

In [ ]:
# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(labels)          # 0-9 integers
y_cat     = to_categorical(y_encoded, num_classes=len(GENRES))  # One-hot for CNN

# Train / Validation / Test split  (70 / 15 / 15)
(X_mel_trainval, X_mel_test,
 X_xgb_trainval, X_xgb_test,
 y_trainval,     y_test,
 y_cat_trainval, y_cat_test) = train_test_split(
    mel_images, xgb_features, y_encoded, y_cat,
    test_size=0.15, random_state=42, stratify=y_encoded
)

(X_mel_train, X_mel_val,
 X_xgb_train, X_xgb_val,
 y_train,     y_val,
 y_cat_train, y_cat_val) = train_test_split(
    X_mel_trainval, X_xgb_trainval, y_trainval, y_cat_trainval,
    test_size=0.176, random_state=42, stratify=y_trainval   # 0.176 ≈ 15% of total
)

# Scale XGB features
scaler = StandardScaler()
X_xgb_train = scaler.fit_transform(X_xgb_train)
X_xgb_val   = scaler.transform(X_xgb_val)
X_xgb_test  = scaler.transform(X_xgb_test)

print(f'Train   : {X_mel_train.shape[0]} samples')
print(f'Val     : {X_mel_val.shape[0]} samples')
print(f'Test    : {X_mel_test.shape[0]} samples')
print(f'Classes : {le.classes_}')

## 🧠 Cell 7: CNN Model — Mel-Spectrogram

In [ ]:
def build_cnn(input_shape=(IMG_H, IMG_W, 1), num_classes=10):
    inp = Input(shape=input_shape)

    # Block 1
    x = Conv2D(32, (3,3), activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.25)(x)

    # Block 2
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.25)(x)

    # Block 3
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)

    # Dense
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)

    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inp, outputs=out, name='GenreCNN')
    return model


cnn_model = build_cnn(num_classes=len(GENRES))
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False,  # Spectrogram flip nahi karna
    zoom_range=0.1
)

history = cnn_model.fit(
    datagen.flow(X_mel_train, y_cat_train, batch_size=BATCH_SIZE),
    validation_data=(X_mel_val, y_cat_val),
    epochs=EPOCHS,
    callbacks=callbacks
)

## 🏋️ Cell 8: CNN Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history = cnn_model.fit(
    X_mel_train, y_cat_train,
    validation_data=(X_mel_val, y_cat_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['accuracy'],     label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('CNN Accuracy'); ax1.legend(); ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('CNN Loss'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

cnn_val_acc = max(history.history['val_accuracy'])
print(f'\n✅ Best CNN Val Accuracy: {cnn_val_acc:.4f}')

## 🌳 Cell 9: XGBoost Training

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_xgb_train, y_train,
    eval_set=[(X_xgb_val, y_val)],
    verbose=50
)

xgb_val_acc = accuracy_score(y_val, xgb_model.predict(X_xgb_val))
print(f'\n✅ XGBoost Val Accuracy: {xgb_val_acc:.4f}')

## 🔗 Cell 10: Stacking Ensemble — Meta-Learner

**Strategy:** CNN aur XGB dono ke probability outputs ko concatenate karke Logistic Regression (meta-learner) train karo.

```
CNN(input)  → probs_cnn  (10 values)  ─┐
                                         ├→ concat (20 values) → LogReg → genre
XGB(input)  → probs_xgb  (10 values)  ─┘
```

In [ ]:
# ── Get probabilities from both base models (Validation set) ──
cnn_val_probs = cnn_model.predict(X_mel_val, verbose=0)       # (n_val, 10)
xgb_val_probs = xgb_model.predict_proba(X_xgb_val)            # (n_val, 10)

# ── Stack horizontally → meta-features ────────────────────────
meta_val = np.hstack([cnn_val_probs, xgb_val_probs])           # (n_val, 20)

# ── Train meta-learner on val set (out-of-fold trick) ─────────
# For proper stacking you'd use cross_val_predict on train set.
# Here we use val set as held-out meta-training data.
meta_learner = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
meta_learner.fit(meta_val, y_val)

print(' Meta-Learner (Logistic Regression) trained on stacked val predictions!')
print(f'   Meta-feature shape: {meta_val.shape}')

## 📈 Cell 11: Final Evaluation on Test Set

In [ ]:
# ── Get test probabilities ─────────────────────────────────────
cnn_test_probs = cnn_model.predict(X_mel_test, verbose=0)
xgb_test_probs = xgb_model.predict_proba(X_xgb_test)

meta_test = np.hstack([cnn_test_probs, xgb_test_probs])

# ── Final predictions ──────────────────────────────────────────
y_pred_cnn   = np.argmax(cnn_test_probs, axis=1)
y_pred_xgb   = xgb_model.predict(X_xgb_test)
y_pred_stack = meta_learner.predict(meta_test)

# ── Accuracy ──────────────────────────────────────────────────
acc_cnn   = accuracy_score(y_test, y_pred_cnn)
acc_xgb   = accuracy_score(y_test, y_pred_xgb)
acc_stack = accuracy_score(y_test, y_pred_stack)

print('='*45)
print(f'  CNN Accuracy          : {acc_cnn:.4f} ({acc_cnn*100:.2f}%)')
print(f'  XGBoost Accuracy      : {acc_xgb:.4f} ({acc_xgb*100:.2f}%)')
print(f'  Stacking Accuracy 🏆  : {acc_stack:.4f} ({acc_stack*100:.2f}%)')
print('='*45)

# ── Classification Report ─────────────────────────────────────
print('\n📋 Stacking Ensemble — Detailed Report:')
print(classification_report(y_test, y_pred_stack, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

def plot_cm(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontweight='bold')

plot_cm(axes[0], y_test, y_pred_cnn,   f'CNN ({acc_cnn*100:.1f}%)')
plot_cm(axes[1], y_test, y_pred_xgb,   f'XGBoost ({acc_xgb*100:.1f}%)')
plot_cm(axes[2], y_test, y_pred_stack, f'Stacking 🏆 ({acc_stack*100:.1f}%)')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
from skimage.transform import resize

def predict_genre(file_path):
    """Koi bhi .wav file do → Genre predict hoga"""
    mel_img, xgb_feat = extract_features(file_path)

    # Reshape for models
    mel_in  = mel_img[np.newaxis, ...]                   # (1, 128, 128, 1)
    xgb_in  = scaler.transform(xgb_feat.reshape(1, -1))  # (1, features)

    # Get probs
    cnn_probs = cnn_model.predict(mel_in, verbose=0)
    xgb_probs = xgb_model.predict_proba(xgb_in)
    meta_in   = np.hstack([cnn_probs, xgb_probs])

    # Meta-learner prediction
    pred_label = meta_learner.predict(meta_in)[0]
    confidence = np.max(meta_learner.predict_proba(meta_in)) * 100

    genre = le.inverse_transform([pred_label])[0]

    print(f'Predicted Genre : {genre.upper()}')
    print(f'Confidence      : {confidence:.1f}%')

    # Top-3 genres
    all_probs = meta_learner.predict_proba(meta_in)[0]
    top3 = np.argsort(all_probs)[::-1][:3]
    print('\n Top-3 Predictions:')
    for i, idx in enumerate(top3, 1):
        print(f'  {i}. {le.classes_[idx]:<12} → {all_probs[idx]*100:.1f}%')

    return genre


# Upload & predict
print('🎧 Apna koi bhi .wav audio file upload karo:')
uploaded = files.upload()

for fname in uploaded.keys():
    print(f'\n🎼 File: {fname}')
    predict_genre(fname)

In [ ]:
import pickle

# Save CNN
cnn_model.save('genre_cnn_model.h5')

# Save XGB + scaler + meta-learner + label encoder
with open('genre_xgb_stack.pkl', 'wb') as f:
    pickle.dump({
        'xgb_model'    : xgb_model,
        'meta_learner' : meta_learner,
        'scaler'       : scaler,
        'label_encoder': le
    }, f)

print('Models saved!')
print('   genre_cnn_model.h5   → CNN model')
print('   genre_xgb_stack.pkl  → XGB + Meta-Learner + Scaler + LabelEncoder')

# Download karo
files.download('genre_cnn_model.h5')
files.download('genre_xgb_stack.pkl')

---
## 📝 Summary

| Component | Details |
|-----------|--------|
| **Dataset** | GTZAN — 10 genres, 1000 audio files |
| **CNN Input** | Mel-Spectrogram (128×128×1) |
| **XGB Input** | MFCC (40) + Chroma + Contrast + ZCR + RMS → ~160 features |
| **CNN Architecture** | 3 Conv Blocks + GAP + Dense |
| **XGBoost** | 500 trees, depth=6, lr=0.05 |
| **Ensemble** | Stacking with Logistic Regression meta-learner |
| **Meta-features** | CNN probs (10) + XGB probs (10) = 20 features |
